# Language Identification — Assignment 2
## Classifier Comparison using FastText Word Vectors

This notebook continues the **language identification (LID)** system built in
Assignment 1 (23 Indian language/script combinations from `IndicCorp v2`). Here we
represent sentences with **pretrained FastText Wikipedia word vectors**
(https://fasttext.cc/docs/en/pretrained-vectors.html) and compare five families of
classifiers, all evaluated with a **from-scratch macro-F1** implementation (no
`sklearn` classes are used anywhere for evaluation):

1. **Logistic Regression** on averaged FastText word vectors
2. **Multi-Layer Perceptron** with `n = 0, 1, 2` hidden layers
3. **LSTM** over the sequence of word vectors
4. **GRU** over the sequence of word vectors
5. **FastText supervised classifier** (`fasttext.train_supervised`)

**Inputs from Assignment 1** (must be present in the working directory, or re-generate
them by re-running the A1 notebook first):

- `train.csv`, `val.csv`, `test.csv` — columns `text`, `language`, `label`
- `label_map.json` — `{id: language_label}`

**Note on sklearn:** `sklearn` is not imported anywhere in this notebook. Metrics
(confusion matrix, precision/recall/F1, macro-F1) are implemented by hand (reused from
Assignment 1). Models 1–4 are implemented with `torch.nn` (linear layers / LSTM / GRU
building blocks — not sklearn), and Model 5 uses Facebook's official `fasttext`
library as explicitly permitted by the assignment.


In [ ]:
# If a package is missing, uncomment and run:
# !pip install -q torch numpy pandas matplotlib tqdm fasttext gensim requests


In [10]:
import os
import re
import json
import zipfile
import gzip
import shutil
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cpu


## Step 1: Load the Assignment-1 dataset

We reuse the exact stratified 80/10/10 split produced in Assignment 1, so results are
directly comparable to the TF-IDF + softmax-regression baseline built there.


In [11]:
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")

with open("label_map.json", "r", encoding="utf-8") as f:
    id2lang = {int(k): v for k, v in json.load(f).items()}
lang2id = {v: k for k, v in id2lang.items()}

N_CLASSES = len(id2lang)
print(f"Languages ({N_CLASSES}):", list(id2lang.values()))
print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)
train_df.head()


Languages (23): ['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'khasi', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'santhali', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab']
Train: (20493, 3)  Val: (2553, 3)  Test: (2576, 3)


,text,language,label
0,2015 ஆம் ஆண்டு இன்னும் ஒரு வாரத்தில் முடிவடைய ...,tam_Taml,20
1,"ਧੂੜ ਮਿੱਟੀ ਤੋਂ ਅਲਰਜੀ ,ਖੂਨ ਦੇ ਚਿੱਟੇ ਸੈੱਲ ਦੀ ਕਮੀ,...",pan_Guru,16
2,సమీర్ వాంఖడేపై వరుస ఆరోపణలు.,tel_Telu,21
3,गसाइगावनि श्रिरामपुरआव पुलिसनि सिग्रोमनायाव गो...,brx_Deva,2
4,গত শুক্রবার ৯৪ বছর বয়সে তিনি মারা যান।,ben_Beng,1


## Step 2: From-scratch evaluation metrics (macro-F1)

Identical logic to Assignment 1: build the confusion matrix by hand, derive per-class
precision/recall/F1, then macro-average. **No `sklearn.metrics` is used anywhere in
this notebook.**


In [12]:
def confusion_matrix_custom(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def precision_recall_f1_per_class(cm):
    n_classes = cm.shape[0]
    precision = np.zeros(n_classes)
    recall = np.zeros(n_classes)
    f1 = np.zeros(n_classes)

    for c in range(n_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp

        precision[c] = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall[c] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1[c] = (2 * precision[c] * recall[c] / (precision[c] + recall[c])
                 if (precision[c] + recall[c]) > 0 else 0.0)

    return precision, recall, f1


def macro_f1_score(y_true, y_pred, n_classes):
    cm = confusion_matrix_custom(y_true, y_pred, n_classes)
    _, _, f1 = precision_recall_f1_per_class(cm)
    return float(np.mean(f1))


def accuracy_score_custom(y_true, y_pred):
    return float(np.mean(np.array(y_true) == np.array(y_pred)))


def classification_report_custom(y_true, y_pred, n_classes, id2label):
    cm = confusion_matrix_custom(y_true, y_pred, n_classes)
    precision, recall, f1 = precision_recall_f1_per_class(cm)
    support = cm.sum(axis=1)
    rows = [{"language": id2label[c], "precision": precision[c], "recall": recall[c],
             "f1": f1[c], "support": support[c]} for c in range(n_classes)]
    report_df = pd.DataFrame(rows)
    return report_df, float(f1.mean()), accuracy_score_custom(y_true, y_pred)


y_train = train_df["language"].map(lang2id).to_numpy()
y_val   = val_df["language"].map(lang2id).to_numpy()
y_test  = test_df["language"].map(lang2id).to_numpy()


## Step 3: FastText Wikipedia word vectors

We use the **pretrained FastText Wikipedia word vectors**
(https://fasttext.cc/docs/en/pretrained-vectors.html, 300-d, one `.zip` per language,
hosted at `dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.<code>.zip`).

Two practical issues we handle explicitly:

1. **Size** — each language file is hundreds of MB to a few GB. We only keep the
   vectors for words that actually occur in our corpus (`train`+`val`+`test`), reading
   the `.vec` file line-by-line instead of loading it fully into memory.
2. **Coverage** — not every IndicCorp v2 language/script has a FastText Wikipedia
   model (e.g. very low-resource languages such as Santali, Bodo, Dogri, or Khasi are
   not covered). For any language code we can't map/download, we fall back to a fixed
   random embedding table for that language's vocabulary (seeded, so it's
   reproducible) and print a warning — this keeps the pipeline runnable end-to-end
   regardless of exactly which 23 languages Assignment 1 sampled.


In [13]:
# Map IndicCorp v2 language/script labels -> FastText Wikipedia ISO 639-1/639-3 codes.
# Extend/edit this table if your Assignment-1 label set differs.
LANG_CODE_MAP = {
    "asm_Beng": "as", "asm": "as",
    "ben_Beng": "bn", "ben": "bn",
    "guj_Gujr": "gu", "guj": "gu",
    "hin_Deva": "hi", "hin": "hi",
    "kan_Knda": "kn", "kan": "kn",
    "kas_Arab": "ks", "kas_Deva": "ks", "kas": "ks",
    "gom_Deva": "gom", "kok_Deva": "gom", "kok": "gom",
    "mai_Deva": "mai", "mai": "mai",
    "mal_Mlym": "ml", "mal": "ml",
    "mar_Deva": "mr", "mar": "mr",
    "mni_Beng": "mni", "mni_Mtei": "mni", "mni": "mni",
    "npi_Deva": "ne", "npi": "ne", "nep": "ne",
    "ory_Orya": "or", "ory": "or",
    "pan_Guru": "pa", "pan": "pa",
    "san_Deva": "sa", "san": "sa",
    "sat_Olck": "sat", "sat_Deva": "sat", "sat": "sat",
    "snd_Arab": "sd", "snd_Deva": "sd", "snd": "sd",
    "tam_Taml": "ta", "tam": "ta",
    "tel_Telu": "te", "tel": "te",
    "urd_Arab": "ur", "urd": "ur",
    "eng_Latn": "en", "eng": "en",
    "khasi": "kha", "brx": "bx", "doi": "doi",
}

FASTTEXT_ZIP_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.{code}.zip"
VEC_DIR = "fasttext_vectors"
os.makedirs(VEC_DIR, exist_ok=True)
EMBED_DIM = 300


def resolve_lang_code(language_label: str):
    if language_label in LANG_CODE_MAP:
        return LANG_CODE_MAP[language_label]
    # fall back: try the part before the first underscore (e.g. "hin_Deva" -> "hin")
    prefix = language_label.split("_")[0].lower()
    return LANG_CODE_MAP.get(prefix)


In [14]:
def download_vec_file(code_: str) -> str:
    """Download+unzip a language's FastText .vec file (cached). Returns local path or None."""
    vec_path = os.path.join(VEC_DIR, f"wiki.{code_}.vec")
    if os.path.exists(vec_path):
        return vec_path

    zip_path = os.path.join(VEC_DIR, f"wiki.{code_}.zip")
    url = FASTTEXT_ZIP_URL.format(code=code_)
    try:
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            with open(zip_path, "wb") as f, tqdm(
                total=total, unit="B", unit_scale=True, desc=f"wiki.{code_}.zip"
            ) as pbar:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
                    pbar.update(len(chunk))
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(VEC_DIR)
        os.remove(zip_path)
        return vec_path if os.path.exists(vec_path) else None
    except Exception as e:
        print(f"  [warn] could not download vectors for '{code_}': {e}")
        return None


def load_vectors_for_vocab(vec_path: str, vocab: set, dim: int = EMBED_DIM) -> dict:
    """Stream a .vec file, keeping only rows whose token is in `vocab`."""
    found = {}
    with open(vec_path, "r", encoding="utf-8", newline="\n", errors="ignore") as f:
        first_line = f.readline()  # header: "<n_words> <dim>"
        for line in f:
            parts = line.rstrip().split(" ")
            token = parts[0]
            if token in vocab:
                try:
                    found[token] = np.asarray(parts[1:dim + 1], dtype=np.float32)
                except ValueError:
                    continue
            if len(found) == len(vocab):
                break
    return found


In [6]:
WORD_RE = re.compile(r"\w+", re.UNICODE)

def tokenize(text: str):
    return WORD_RE.findall(text.lower())


# Build per-language vocabularies (from train+val+test, so val/test OOV handling is
# consistent — this mirrors what the A1 TF-IDF vectorizer already implicitly assumed).
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
full_df["tokens"] = full_df["text"].apply(tokenize)

lang_vocab = {lang: set() for lang in lang2id}
for lang, toks in zip(full_df["language"], full_df["tokens"]):
    lang_vocab[lang].update(toks)

print({lang: len(v) for lang, v in lang_vocab.items()})


{'asm_Beng': 2105, 'ben_Beng': 1586, 'brx_Deva': 1483, 'doi_Deva': 1095, 'gom_Deva': 1802, 'guj_Gujr': 1865, 'hin_Deva': 1417, 'kan_Knda': 1706, 'kas_Arab': 3735, 'khasi': 6372, 'mai_Deva': 1806, 'mal_Mlym': 1793, 'mar_Deva': 1694, 'mni_Mtei': 2577, 'npi_Deva': 1844, 'ory_Orya': 1678, 'pan_Guru': 1740, 'san_Deva': 1650, 'santhali': 6957, 'snd_Deva': 5880, 'tam_Taml': 1411, 'tel_Telu': 1199, 'urd_Arab': 5770}


In [15]:
# Build a single global embedding table: word -> 300-d vector.
# For languages with a resolvable FastText code, we download+load the real vectors.
# For anything unresolved/unavailable, we substitute a fixed random table (documented
# fallback) so the rest of the pipeline is unaffected.
rng = np.random.RandomState(RANDOM_SEED)
embedding_table = {}          # token -> np.ndarray(300,)
languages_with_pretrained = []
languages_fallback = []

for lang, vocab in lang_vocab.items():
    code_ = resolve_lang_code(lang)
    vec_path = download_vec_file(code_) if code_ else None

    if vec_path is not None:
        found = load_vectors_for_vocab(vec_path, vocab, EMBED_DIM)
        embedding_table.update(found)
        missing = vocab - found.keys()
        for tok in missing:  # OOV within a covered language -> small random vector
            embedding_table.setdefault(tok, rng.normal(0, 0.1, EMBED_DIM).astype(np.float32))
        languages_with_pretrained.append(lang)
    else:
        for tok in vocab:
            embedding_table.setdefault(tok, rng.normal(0, 0.1, EMBED_DIM).astype(np.float32))
        languages_fallback.append(lang)

print(f"Languages using real FastText wiki vectors ({len(languages_with_pretrained)}):",
      languages_with_pretrained)
print(f"Languages using random fallback vectors ({len(languages_fallback)}):",
      languages_fallback)
print("Total vocabulary size with embeddings:", len(embedding_table))

OOV_VECTOR = rng.normal(0, 0.1, EMBED_DIM).astype(np.float32)  # for unseen tokens at inference time

def get_vector(token: str) -> np.ndarray:
    return embedding_table.get(token, OOV_VECTOR)


wiki.as.zip:   0%|          | 0.00/2.40G [00:00<?, ?B/s]

wiki.bn.zip:   0%|          | 0.00/2.74G [00:00<?, ?B/s]

KeyboardInterrupt: 

## Step 4: Featurization

Two representations are built from the same embedding table:

- **Averaged sentence vectors** (mean of word vectors) — used by the Logistic
  Regression and MLP models.
- **Padded sequences of word vectors** — used by the LSTM and GRU models.


In [ ]:
def sentence_to_avg_vector(tokens):
    if not tokens:
        return np.zeros(EMBED_DIM, dtype=np.float32)
    vecs = np.stack([get_vector(t) for t in tokens])
    return vecs.mean(axis=0)


def build_avg_matrix(df):
    return np.stack([sentence_to_avg_vector(toks) for toks in df["tokens"]]).astype(np.float32)


train_tokens = train_df["text"].apply(tokenize)
val_tokens   = val_df["text"].apply(tokenize)
test_tokens  = test_df["text"].apply(tokenize)

train_df["tokens"], val_df["tokens"], test_df["tokens"] = train_tokens, val_tokens, test_tokens

X_train_avg = build_avg_matrix(train_df)
X_val_avg   = build_avg_matrix(val_df)
X_test_avg  = build_avg_matrix(test_df)

print("X_train_avg:", X_train_avg.shape, "X_val_avg:", X_val_avg.shape, "X_test_avg:", X_test_avg.shape)


In [ ]:
MAX_LEN = int(np.percentile(train_df["tokens"].apply(len), 95))
MAX_LEN = max(MAX_LEN, 4)
print("Sequence length capped at:", MAX_LEN)

def build_seq_tensor(df):
    n = len(df)
    X = np.zeros((n, MAX_LEN, EMBED_DIM), dtype=np.float32)
    lengths = np.zeros(n, dtype=np.int64)
    for i, toks in enumerate(df["tokens"]):
        toks = toks[:MAX_LEN]
        L = max(len(toks), 1)
        lengths[i] = L
        for j, t in enumerate(toks):
            X[i, j] = get_vector(t)
    return X, lengths

X_train_seq, len_train = build_seq_tensor(train_df)
X_val_seq,   len_val   = build_seq_tensor(val_df)
X_test_seq,  len_test  = build_seq_tensor(test_df)

print("X_train_seq:", X_train_seq.shape)


## Step 5: Generic PyTorch training utilities

All four neural models (LR, MLP, LSTM, GRU) share the same training loop: cross-entropy
loss, Adam optimizer, and macro-F1 (our own implementation, not `sklearn`) tracked on
the validation set after every epoch, keeping the best-validation-macro-F1 weights.


In [ ]:
class ArrayDataset(Dataset):
    def __init__(self, *arrays):
        self.arrays = arrays
    def __len__(self):
        return len(self.arrays[0])
    def __getitem__(self, idx):
        return tuple(torch.as_tensor(a[idx]) for a in self.arrays)


def train_torch_model(model, train_arrays, y_train, val_arrays, y_val,
                       n_classes, epochs=15, batch_size=128, lr=1e-3,
                       weight_decay=1e-5, verbose=True, model_name="model"):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    train_ds = ArrayDataset(*train_arrays, y_train)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    val_tensors = [torch.as_tensor(a).to(DEVICE) for a in val_arrays]

    best_val_f1, best_state = -1.0, None
    history = {"train_loss": [], "val_macro_f1": []}

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_losses = []
        for batch in train_loader:
            *inputs, yb = batch
            inputs = [x.to(DEVICE) for x in inputs]
            yb = yb.to(DEVICE).long()

            opt.zero_grad()
            logits = model(*inputs)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            epoch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_logits = model(*val_tensors)
            val_pred = val_logits.argmax(dim=1).cpu().numpy()
        val_f1 = macro_f1_score(y_val, val_pred, n_classes)

        history["train_loss"].append(float(np.mean(epoch_losses)))
        history["val_macro_f1"].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

        if verbose:
            print(f"[{model_name}] epoch {epoch:02d}/{epochs}  "
                  f"train_loss={history['train_loss'][-1]:.4f}  val_macro_F1={val_f1:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


def evaluate_torch_model(model, test_arrays, y_test, n_classes, id2label):
    model.eval()
    test_tensors = [torch.as_tensor(a).to(DEVICE) for a in test_arrays]
    with torch.no_grad():
        logits = model(*test_tensors)
        pred = logits.argmax(dim=1).cpu().numpy()
    report_df, macro_f1, acc = classification_report_custom(y_test, pred, n_classes, id2label)
    return pred, report_df, macro_f1, acc


## Model 1: Logistic Regression on averaged FastText vectors

A single linear layer + softmax (via `CrossEntropyLoss`) over the 300-d averaged
sentence embedding — the neural equivalent of multinomial logistic regression.


In [ ]:
class LogisticRegressionTorch(nn.Module):
    def __init__(self, input_dim, n_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, n_classes)

    def forward(self, x):
        return self.linear(x)


lr_model = LogisticRegressionTorch(EMBED_DIM, N_CLASSES)
lr_model, lr_history = train_torch_model(
    lr_model, (X_train_avg,), y_train, (X_val_avg,), y_val,
    n_classes=N_CLASSES, epochs=20, batch_size=128, lr=1e-3, model_name="LogReg",
)
lr_pred, lr_report, lr_test_f1, lr_test_acc = evaluate_torch_model(
    lr_model, (X_test_avg,), y_test, N_CLASSES, id2lang
)
print(f"\nLogistic Regression -> Test accuracy={lr_test_acc:.4f}  Test macro-F1={lr_test_f1:.4f}")


## Model 2: Multi-Layer Perceptron (n = 0, 1, 2 hidden layers)

`hidden_dims=[]` (n=0) reduces to the same architecture as the logistic regression
model above (softmax over a linear projection) — included explicitly so the two are
compared under the exact same training procedure.


In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims, n_classes, dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, n_classes)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


mlp_configs = {
    "MLP (n=0 hidden layers)": [],
    "MLP (n=1 hidden layer)":  [256],
    "MLP (n=2 hidden layers)": [256, 128],
}

mlp_results = {}
for name, hidden_dims in mlp_configs.items():
    model = MLPClassifier(EMBED_DIM, hidden_dims, N_CLASSES)
    model, hist = train_torch_model(
        model, (X_train_avg,), y_train, (X_val_avg,), y_val,
        n_classes=N_CLASSES, epochs=20, batch_size=128, lr=1e-3, model_name=name,
    )
    pred, report_df, test_f1, test_acc = evaluate_torch_model(
        model, (X_test_avg,), y_test, N_CLASSES, id2lang
    )
    mlp_results[name] = {"model": model, "history": hist, "report": report_df,
                          "test_macro_f1": test_f1, "test_acc": test_acc}
    print(f"{name} -> Test accuracy={test_acc:.4f}  Test macro-F1={test_f1:.4f}\n")


## Model 3 & 4: LSTM and GRU over the word-vector sequence

Both models consume the padded `(batch, seq_len, 300)` FastText embedding sequence,
run a single recurrent layer, and classify from the final hidden state.


In [ ]:
class RecurrentClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_classes, cell="lstm", num_layers=1, dropout=0.2):
        super().__init__()
        rnn_cls = nn.LSTM if cell == "lstm" else nn.GRU
        self.rnn = rnn_cls(input_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, lengths):
        lengths_cpu = lengths.detach().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths_cpu, batch_first=True, enforce_sorted=False
        )
        packed_out, state = self.rnn(packed)
        h_n = state[0] if isinstance(state, tuple) else state  # LSTM: (h,c); GRU: h
        last_hidden = h_n[-1]  # (batch, hidden_dim), final layer's hidden state
        return self.fc(self.dropout(last_hidden))


In [ ]:
lstm_model = RecurrentClassifier(EMBED_DIM, hidden_dim=128, n_classes=N_CLASSES, cell="lstm")
lstm_model, lstm_history = train_torch_model(
    lstm_model, (X_train_seq, len_train), y_train, (X_val_seq, len_val), y_val,
    n_classes=N_CLASSES, epochs=12, batch_size=128, lr=1e-3, model_name="LSTM",
)
lstm_pred, lstm_report, lstm_test_f1, lstm_test_acc = evaluate_torch_model(
    lstm_model, (X_test_seq, len_test), y_test, N_CLASSES, id2lang
)
print(f"\nLSTM -> Test accuracy={lstm_test_acc:.4f}  Test macro-F1={lstm_test_f1:.4f}")


In [ ]:
gru_model = RecurrentClassifier(EMBED_DIM, hidden_dim=128, n_classes=N_CLASSES, cell="gru")
gru_model, gru_history = train_torch_model(
    gru_model, (X_train_seq, len_train), y_train, (X_val_seq, len_val), y_val,
    n_classes=N_CLASSES, epochs=12, batch_size=128, lr=1e-3, model_name="GRU",
)
gru_pred, gru_report, gru_test_f1, gru_test_acc = evaluate_torch_model(
    gru_model, (X_test_seq, len_test), y_test, N_CLASSES, id2lang
)
print(f"\nGRU -> Test accuracy={gru_test_acc:.4f}  Test macro-F1={gru_test_f1:.4f}")


## Model 5: FastText supervised classifier

Uses Facebook's official `fasttext` library
(https://fasttext.cc/docs/en/supervised-tutorial.html), which is explicitly allowed by
the assignment. Data is converted to FastText's `__label__<lang> <text>` format; the
model learns its own embeddings + classifier jointly. Evaluation still goes through our
own `macro_f1_score` (not `fasttext`'s built-in `test()` precision/recall, and not
`sklearn`), so all five models are scored identically.


In [ ]:
# !pip install -q fasttext
import fasttext

def clean_for_fasttext(text: str) -> str:
    return " ".join(text.split())  # fastText treats newlines as example boundaries

def write_fasttext_file(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            f.write(f"__label__{row['language']} {clean_for_fasttext(row['text'])}\n")

write_fasttext_file(train_df, "ft_train.txt")
write_fasttext_file(val_df,   "ft_val.txt")
write_fasttext_file(test_df,  "ft_test.txt")

ft_model = fasttext.train_supervised(
    input="ft_train.txt",
    dim=EMBED_DIM,
    epoch=25,
    lr=0.5,
    wordNgrams=2,
    minCount=1,
    loss="softmax",
    seed=RANDOM_SEED,
    verbose=2,
)
ft_model.save_model("fasttext_lid_model.bin")


In [ ]:
def fasttext_predict_labels(model, texts):
    preds = []
    for t in texts:
        label, _ = model.predict(clean_for_fasttext(t), k=1)
        lang = label[0].replace("__label__", "")
        preds.append(lang2id.get(lang, -1))
    return np.array(preds)

ft_test_pred = fasttext_predict_labels(ft_model, test_df["text"].tolist())

# Any unmapped prediction (shouldn't normally happen) is treated as a miss.
valid_mask = ft_test_pred != -1
ft_report, ft_test_f1, ft_test_acc = classification_report_custom(
    y_test[valid_mask], ft_test_pred[valid_mask], N_CLASSES, id2lang
)
print(f"FastText supervised -> Test accuracy={ft_test_acc:.4f}  Test macro-F1={ft_test_f1:.4f}")


## Step 6: Compare all classifiers

Macro-F1 (computed entirely with our own `macro_f1_score` implementation) on the held
-out test set, for every model.


In [ ]:
results = {
    "Logistic Regression (avg. FastText vec.)": lr_test_f1,
    "MLP (n=0 hidden layers)": mlp_results["MLP (n=0 hidden layers)"]["test_macro_f1"],
    "MLP (n=1 hidden layer)":  mlp_results["MLP (n=1 hidden layer)"]["test_macro_f1"],
    "MLP (n=2 hidden layers)": mlp_results["MLP (n=2 hidden layers)"]["test_macro_f1"],
    "LSTM": lstm_test_f1,
    "GRU": gru_test_f1,
    "FastText supervised": ft_test_f1,
}

results_df = pd.DataFrame(
    sorted(results.items(), key=lambda kv: kv[1], reverse=True),
    columns=["Model", "Test macro-F1"],
)
results_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(results_df["Model"][::-1], results_df["Test macro-F1"][::-1], color="tab:blue")
ax.set_xlabel("Test macro-F1")
ax.set_xlim(0, 1)
ax.set_title("Language Identification — Classifier Comparison (macro-F1)")
for i, v in enumerate(results_df["Test macro-F1"][::-1]):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center")
fig.tight_layout()
plt.show()

results_df.to_csv("classifier_comparison_macro_f1.csv", index=False)


## Discussion

- **Logistic Regression vs. MLP (n=0):** these two use the same underlying softmax
  regression over averaged FastText vectors, trained with the same optimizer/loop —
  their scores should be nearly identical, and any (small) gap is just optimization
  noise. This empirically confirms that "MLP with 0 hidden layers" *is* logistic
  regression.
- **Averaging word vectors** (LR/MLP) discards word order, which can hurt on
  morphologically rich / low-resource Indic scripts where a handful of function words
  or character patterns carry a lot of the signal that our from-scratch TF-IDF
  approach in Assignment 1 captured more directly.
- **LSTM/GRU** preserve sequence order and can pick up on sub-word / script-level
  cues over time, typically closing (or reversing) the gap versus averaging — at the
  cost of slower training and more parameters.
- **FastText supervised** jointly learns subword (character n-gram) embeddings *and*
  the classifier end-to-end for this exact label set, which is usually the strongest
  and fastest baseline for language ID specifically, since language ID is close to the
  n-gram-classification task FastText was designed for.
- **Coverage caveat:** any Assignment-1 language without official FastText Wikipedia
  vectors falls back to fixed random embeddings for models 1–4 (see Step 3's printed
  warnings) — expect the LR/MLP/LSTM/GRU macro-F1 to be pulled down specifically on
  those languages, while the pure FastText-supervised model (which learns its own
  embeddings from scratch) is unaffected by this limitation.
